In [83]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, precision_score, recall_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss,
    precision_score, recall_score, f1_score,
)

In [84]:
features = pd.read_parquet('data/features.parquet')
features.head(20)

,id1,id2,target,fz_ratio,fz_partial_ratio,fz_token_sort_ratio,fz_token_set_ratio,jaccard,numbers_jaccard,brand_common,weight_rel_diff,article_equal,partnumber_equal,brand_equal,type_similarity,color_equal,material_equal,size_equal
0,32,188978721454,0.0,78.301887,80.000000,78.301887,87.179487,0.545455,0.500000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,34,197568543106,0.0,80.733945,84.848485,78.899083,86.363636,0.555556,NaN,1.0,NaN,NaN,NaN,1.0,100.000000,0.0,NaN,NaN
2,37,661425040668,0.0,44.444444,64.000000,94.444444,100.000000,0.750000,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,97,274877917548,0.0,57.435897,60.540541,55.384615,61.702128,0.250000,0.666667,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,154,283467899529,0.0,57.894737,68.965517,52.631579,55.263158,0.153846,0.500000,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
5,192,257698056754,0.0,38.383838,50.793651,52.525253,65.517241,0.230769,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,197,558345884222,0.0,48.062016,51.612903,49.612403,53.448276,0.153846,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,415,609885460234,0.0,43.396226,64.406780,35.849057,48.000000,0.166667,0.000000,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
8,476,841813632218,1.0,68.354430,79.411765,75.949367,83.544304,0.333333,1.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,525,13777,1.0,44.274809,66.666667,44.274809,44.274809,0.095238,1.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
item_ids = pd.concat([features['id1'], features['id2']], ignore_index=True)
item_counts = item_ids.value_counts()

In [86]:
repeated_ids = item_counts[item_counts > 1].index

In [87]:
repeat_mask = features['id1'].isin(repeated_ids) | features['id2'].isin(repeated_ids)
repeat_mask.sum()

np.int64(34775)

## Train, validation и test



In [88]:
train_repeated = features[repeat_mask]
remaining = features[~repeat_mask]


train_remaining, test = train_test_split(
    remaining, test_size=0.2, random_state=42,
    stratify=remaining['target']
)


train_remaining, validation = train_test_split(
    train_remaining, test_size=0.2, random_state=42,
    stratify=train_remaining['target']
)

train = pd.concat([train_repeated, train_remaining], ignore_index=True)


In [89]:
item_ids = pd.concat([test['id1'], test['id2']], ignore_index=True)
item_counts = item_ids.value_counts()
print((item_counts > 1).sum())

0


In [90]:
train_ids = set(train['id1']) | set(train['id2'])
validation_ids = set(validation['id1']) | set(validation['id2'])
test_ids = set(test['id1']) | set(test['id2'])

print('Общих товаров train / validation:', len(train_ids & validation_ids))
print('Общих товаров train / test:', len(train_ids & test_ids))
print('Общих товаров validation / test:', len(validation_ids & test_ids))


Общих товаров train / validation: 0
Общих товаров train / test: 0
Общих товаров validation / test: 0


In [91]:
print('Доля дублей train:', train['target'].mean())
print('Доля дублей validation:', validation['target'].mean())
print('Доля дублей test:', test['target'].mean())


Доля дублей train: 0.25268012509278526
Доля дублей validation: 0.2652386619066508
Доля дублей test: 0.26524721953578334


In [92]:
X_train = train.drop(columns=['target', 'id1', 'id2'])
X_validation = validation.drop(columns=['target', 'id1', 'id2'])
X_test = test.drop(columns=['target', 'id1', 'id2'])

y_train = train['target']
y_validation = validation['target']
y_test = test['target']


In [93]:
print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_validation.shape, y_validation.shape)
print('Test:', X_test.shape, y_test.shape)


Train: (246537, 15) (246537,)
Validation: (52941, 15) (52941,)
Test: (66176, 15) (66176,)


## Обучение



In [94]:
model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='Logloss',
    random_seed=42,
    verbose=50
)

model.fit(
    X_train, y_train,
    eval_set=(X_validation, y_validation),
    early_stopping_rounds=50,
    use_best_model=True
)


0:	learn: 0.6741793	test: 0.6748199	best: 0.6748199 (0)	total: 132ms	remaining: 2m 11s
50:	learn: 0.4983624	test: 0.5091146	best: 0.5091146 (50)	total: 4.41s	remaining: 1m 22s
100:	learn: 0.4880539	test: 0.5005952	best: 0.5005952 (100)	total: 9.17s	remaining: 1m 21s
150:	learn: 0.4829142	test: 0.4966927	best: 0.4966927 (150)	total: 12.9s	remaining: 1m 12s
200:	learn: 0.4796128	test: 0.4945778	best: 0.4945778 (200)	total: 16.5s	remaining: 1m 5s
250:	learn: 0.4768068	test: 0.4927543	best: 0.4927543 (250)	total: 19.3s	remaining: 57.5s
300:	learn: 0.4741904	test: 0.4913155	best: 0.4913155 (300)	total: 22.5s	remaining: 52.2s
350:	learn: 0.4717530	test: 0.4902758	best: 0.4902758 (350)	total: 25.4s	remaining: 46.9s
400:	learn: 0.4696594	test: 0.4894827	best: 0.4894827 (400)	total: 28.4s	remaining: 42.4s
450:	learn: 0.4675955	test: 0.4888275	best: 0.4888275 (450)	total: 31.5s	remaining: 38.3s
500:	learn: 0.4658331	test: 0.4882034	best: 0.4882034 (500)	total: 35.1s	remaining: 34.9s
550:	learn: 

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

## Метрики validation


In [95]:
validation_proba = model.predict_proba(X_validation)[:, 1]

print('ROC-AUC:', roc_auc_score(y_validation, validation_proba))
print('Average Precision:', average_precision_score(y_validation, validation_proba))
print('Logloss:', log_loss(y_validation, validation_proba))

validation_pred = (validation_proba >= 0.3).astype(int)

print('Precision:', precision_score(y_validation, validation_pred, zero_division=0))
print('Recall:', recall_score(y_validation, validation_pred, zero_division=0))
print('F1:', f1_score(y_validation, validation_pred, zero_division=0))


ROC-AUC: 0.7593852811527187
Average Precision: 0.5640967711571114
Logloss: 0.4854510831654467
Precision: 0.48943317859760566
Recall: 0.5706452072354365
F1: 0.5269283882422569


## Случайное предсказание


In [96]:
dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(X_train, y_train)

dummy_proba = dummy.predict_proba(X_validation)[:, 1]

threshold = 0.2
dummy_pred = (dummy_proba >= threshold).astype(int)

print('ROC-AUC:', roc_auc_score(y_validation, dummy_proba))
print('Average Precision:', average_precision_score(y_validation, dummy_proba))
print('Logloss:', log_loss(y_validation, dummy_proba))
print('Precision:', precision_score(y_validation, dummy_pred, zero_division=0))
print('Recall:', recall_score(y_validation, dummy_pred, zero_division=0))
print('F1:', f1_score(y_validation, dummy_pred, zero_division=0))

ROC-AUC: 0.5018286550154416
Average Precision: 0.2659590452519996
Logloss: 13.762233611144747
Precision: 0.2680745528333083
Recall: 0.25402364335564737
F1: 0.26086002632733657


## Метрики test



In [97]:
y_proba = model.predict_proba(X_test)[:, 1]


In [98]:
print('ROC-AUC:', roc_auc_score(y_test, y_proba))
print('Logloss:', log_loss(y_test, y_proba))

ROC-AUC: 0.7638921760698982
Logloss: 0.48288286066290637


## Метрики train


In [99]:
train_proba = model.predict_proba(X_train)[:, 1]

print('ROC-AUC:', roc_auc_score(y_train, train_proba))
print('Logloss:', log_loss(y_train, train_proba))

ROC-AUC: 0.7937895556443164
Logloss: 0.4512634062487514
